# Chapter 8 · Statistical Mechanics from Simulations — live notebook

This notebook runs entirely in your browser via the Pyodide kernel.
All state is saved to your browser's local storage; nothing is sent to a server.

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch08-statmech/>

Two short experiments: sampling a Boltzmann distribution with Metropolis Monte Carlo, and extracting a diffusion coefficient from a 1D random walk via the Einstein relation.

Only Pyodide-compatible packages are used (numpy, scipy, matplotlib, ipywidgets).


## 8.1 Metropolis sampling of a 1D Boltzmann distribution

Target: $\rho(x) \propto \exp(-\beta U(x))$ with a double-well potential. We plot the histogram against the analytic distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

def U(x):
    return (x ** 2 - 1) ** 2

beta = 5.0
x = 0.0
samples = []
for _ in range(200_000):
    xp = x + rng.normal(0, 0.4)
    if rng.random() < np.exp(-beta * (U(xp) - U(x))):
        x = xp
    samples.append(x)
samples = np.array(samples[5000:])  # discard burn-in

grid = np.linspace(-2, 2, 400)
p = np.exp(-beta * U(grid))
p /= 0.5 * np.sum((p[1:] + p[:-1]) * np.diff(grid))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(samples, bins=80, density=True, alpha=0.6, label='Metropolis')
ax.plot(grid, p, 'k', lw=2, label='analytic Boltzmann')
ax.set_xlabel('x')
ax.set_ylabel('rho(x)')
ax.set_title('Double-well at beta = 5')
ax.legend()
plt.show()


## 8.2 Diffusion from a random walk — the Einstein relation

Mean-squared displacement of an unbiased walk grows linearly in time, with slope $2 D$ in 1D.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)
n_walkers = 2000
n_steps = 1500
step_size = 1.0
steps = rng.normal(0.0, step_size, size=(n_walkers, n_steps))
x = np.cumsum(steps, axis=1)
msd = (x ** 2).mean(axis=0)
t = np.arange(1, n_steps + 1)
D_fit = np.polyfit(t, msd, 1)[0] / 2

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(t, msd, label='measured MSD')
ax.plot(t, 2 * D_fit * t, 'r--', label=f'fit: D = {D_fit:.3f}')
ax.set_xlabel('step')
ax.set_ylabel('<x^2>')
ax.legend()
ax.set_title('Einstein relation in 1D')
plt.show()
